# RAG (Retrieval Augmented Generation)

## Módulo 5 — Agentes Avanzados | Semana 11

---

## Objetivos de Aprendizaje

| # | Objetivo |
|---|----------|
| 1 | Entender **qué problema resuelve RAG** y por qué Claude necesita tus documentos privados |
| 2 | Generar y trabajar con **embeddings** (vectores semánticos del texto) |
| 3 | Implementar **estrategias de chunking** para dividir documentos de forma óptima |
| 4 | Usar **vector stores** (ChromaDB y pgvector) para almacenar y buscar embeddings |
| 5 | Construir un **pipeline RAG completo** integrado con un agente |
| 6 | **Evaluar la calidad** del RAG con métricas de retrieval y comparaciones |

---

> **Contexto**: Seguimos trabajando con **Klimbook** (app de escalada). En esta lección el agente va a responder preguntas técnicas sobre Klimbook buscando en la documentación interna del proyecto, no inventando respuestas genéricas.

---

## 11.1 El Problema que RAG Resuelve

Claude fue entrenado con datos públicos de internet hasta cierta fecha. **No conoce**:

- El código de Klimbook
- Tu documentación interna
- Los datos de tus usuarios
- Tus decisiones de arquitectura
- Tus changelogs privados

Si le preguntas *"¿cómo funciona la autenticación en Klimbook?"*, Claude va a inventar una respuesta genérica sobre autenticación en FastAPI. No sabe que **tu** servicio de auth usa RS256 para JWT, que los refresh tokens se guardan en Redis, ni que el endpoint es `/auth/login`.

### Sin RAG vs Con RAG

```
Sin RAG:
  Usuario: "Como funciona la autenticacion en Klimbook?"
  Claude:  "Tipicamente, la autenticacion en FastAPI se implementa
            con OAuth2 y JWT tokens. Puedes usar python-jose..."
  -> Respuesta GENERICA. No sabe nada de tu implementacion.

Con RAG:
  Usuario: "Como funciona la autenticacion en Klimbook?"
  Sistema: 1. Busca en tu documentacion los fragmentos relevantes
           2. Encuentra: auth-service README, login endpoint docs
           3. Los inyecta en el prompt como contexto
  Claude:  "La autenticacion en Klimbook usa el auth-service con JWT
            firmados con RS256. El endpoint POST /auth/login recibe
            email y password, valida con bcrypt, y retorna un token
            + refresh_token. Los refresh tokens se almacenan en Redis
            con TTL de 7 dias..."
  -> Respuesta ESPECIFICA basada en TUS documentos reales.
```

La diferencia es enorme. **RAG convierte a Claude de un asistente genérico a un experto en TU proyecto.**

---

## 11.2 El Pipeline de RAG (Paso a Paso)

RAG tiene **dos fases**: indexación (se hace una vez) y consulta (se hace en cada pregunta).

```
FASE 1: INDEXACION (offline, una sola vez)
==========================================

Tus documentos                     Vector Store
    |                                    |
    v                                    |
[README.md]                              |
[API_DOCS.md]  ---> Chunking --->  Embeddings ---> Guardar
[CHANGELOG.md]      (dividir)      (vectorizar)    (indexar)
[codigo.py]


FASE 2: CONSULTA (online, cada pregunta)
=========================================

Pregunta del usuario
    |
    v
Embedding de la pregunta
    |
    v
Buscar en Vector Store (similitud coseno)
    |
    v
Top-K documentos mas relevantes
    |
    v
Inyectar en el prompt como contexto
    |
    v
Claude genera respuesta basada en los documentos
```

Vamos a ver cada paso en detalle en las secciones siguientes.

---

## 11.3 Embeddings: Representaciones Numéricas del Texto

Un **embedding** es un vector numérico (una lista de números) que captura el **significado semántico** del texto. Textos con significado similar tienen vectores cercanos en el espacio.

```
Texto: "Como escalar una ruta de 5.11a"
Embedding: [0.023, -0.156, 0.891, 0.034, ..., -0.445]
           (vector de 1024 dimensiones)

Texto: "Tecnica para subir una via de grado 5.11a"
Embedding: [0.025, -0.148, 0.887, 0.031, ..., -0.441]
           (vector MUY similar al anterior -- mismo significado)

Texto: "Receta de pastel de chocolate"
Embedding: [0.567, 0.234, -0.012, 0.789, ..., 0.123]
           (vector MUY diferente -- significado completamente distinto)
```

Los dos primeros textos hablan de lo mismo (escalar una ruta 5.11a) aunque usen **palabras diferentes**. Sus vectores son casi idénticos. El tercero habla de algo completamente distinto, así que su vector es muy diferente.

### ¿Cómo se mide la similitud?

Se usa **distancia coseno** (cosine similarity). Es un número entre 0 y 1:

| Valor | Significado |
|-------|-------------|
| **1.0** | Textos idénticos en significado |
| **~0.6-0.8** | Textos muy relacionados |
| **~0.3-0.5** | Algo relacionados |
| **~0.0** | Completamente diferentes |

In [ ]:
import numpy as np


def cosine_similarity(vec_a, vec_b):
    """
    Calcula la similitud coseno entre dos vectores.
    
    La formula es: cos(A,B) = (A . B) / (||A|| * ||B||)
    
    Donde:
    - A . B es el producto punto (sum of element-wise multiplication)
    - ||A|| es la norma (longitud) del vector A
    
    Ejemplo:
      vec_a = [1, 0, 1], vec_b = [1, 0, 1] -> similarity = 1.0 (identicos)
      vec_a = [1, 0, 0], vec_b = [0, 0, 1] -> similarity = 0.0 (sin relacion)
    """
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)


# Demo: vectores simples para entender la intuicion
v1 = np.array([1, 0, 1])
v2 = np.array([1, 0, 1])  # identico a v1
v3 = np.array([0, 1, 0])  # ortogonal (sin relacion)
v4 = np.array([0.9, 0.1, 0.95])  # muy similar a v1

print("=== Similitud Coseno ===")
print(f"  v1 vs v2 (identicos):     {cosine_similarity(v1, v2):.3f}")
print(f"  v1 vs v3 (sin relacion):  {cosine_similarity(v1, v3):.3f}")
print(f"  v1 vs v4 (muy similar):   {cosine_similarity(v1, v4):.3f}")

### Modelos de Embedding

Los embeddings los genera un **modelo especializado** (no Claude). Estas son las opciones principales:

| Modelo | Dimensiones | Costo | Nota |
|--------|-------------|-------|------|
| **Voyage AI** (voyage-3) | 1024 | $0.06/M tokens | Recomendado por Anthropic |
| OpenAI ada-002 | 1536 | $0.10/M tokens | Popular, buena calidad |
| OpenAI text-embedding-3-small | 1536 | $0.02/M tokens | Más barato que ada |
| **sentence-transformers** | 384-768 | Gratis (local) | Corre en tu máquina, sin API |

Para este track usaremos **sentence-transformers** porque es gratis y corre localmente. En producción, Voyage AI es la opción recomendada por Anthropic por su calidad superior.

> **Nota**: más dimensiones ≠ mejor. Un modelo de 384 dims bien entrenado puede superar a uno de 1536 dims malo. Lo que importa es la **calidad del entrenamiento**, no el tamaño del vector.

In [ ]:
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer

# Cargar el modelo (se descarga la primera vez, ~90MB).
# all-MiniLM-L6-v2 es bueno para busqueda semantica en ingles y espanol.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generar embeddings
texts = [
    "How does authentication work in Klimbook?",
    "The auth service uses JWT tokens with RS256 signing.",
    "Recipe for chocolate cake",
]

# model.encode() convierte texto a vectores.
# Retorna un numpy array de shape (n_textos, n_dimensiones).
embeddings = model.encode(texts)

print(f"Shape: {embeddings.shape}")  # (3, 384) -> 3 textos, 384 dimensiones

# Verificar similitud entre los textos
sim_auth = cosine_similarity(embeddings[0], embeddings[1])
sim_cake = cosine_similarity(embeddings[0], embeddings[2])

print(f"\nAuth question vs Auth doc:  {sim_auth:.3f}")  # ~0.65 (relacionados)
print(f"Auth question vs Cake:      {sim_cake:.3f}")  # ~0.08 (sin relacion)

### Batch Embedding

Cuando indexas muchos documentos, es más eficiente embedizar **todos de una vez** en lugar de uno por uno:

In [ ]:
# MAL: uno por uno (lento -- una llamada al modelo por texto)
# for text in thousands_of_texts:
#     emb = model.encode([text])

# BIEN: en batch (rapido -- una sola llamada que procesa 64 textos a la vez)
# all_embeddings = model.encode(thousands_of_texts, batch_size=64)

# Demo con nuestros textos:
sample_texts = [
    "Klimbook authentication uses RS256 JWT",
    "Routes are graded from 5.6 to 5.15d",
    "The API is built with FastAPI and PostgreSQL",
    "Deploy with Docker Compose on a VPS",
]

batch_embeddings = model.encode(sample_texts, batch_size=2)
print(f"Batch shape: {batch_embeddings.shape}")  # (4, 384)

# Comparar todos contra todos
print("\n=== Matriz de Similitud ===")
for i, t1 in enumerate(sample_texts):
    for j, t2 in enumerate(sample_texts):
        if j > i:
            sim = cosine_similarity(batch_embeddings[i], batch_embeddings[j])
            print(f"  [{i}] vs [{j}]: {sim:.3f}  ({t1[:30]}... vs {t2[:30]}...)")

---

## 11.4 Chunking: Dividir Documentos en Fragmentos

No puedes embedizar un documento de 10,000 palabras como una sola pieza. El embedding capturaría un *"significado promedio"* del documento completo, demasiado vago para ser útil en búsquedas.

Necesitas dividirlo en **chunks** (fragmentos) de tamaño manejable. Cada chunk se embediza por separado, y cuando buscas, recuperas los chunks más relevantes -- no documentos enteros.

### El problema ilustrado

```
Documento: README.md de Klimbook (5000 palabras)
  - Seccion 1: Que es Klimbook       (500 palabras)
  - Seccion 2: Tech Stack            (300 palabras)
  - Seccion 3: Autenticacion         (800 palabras)
  - Seccion 4: Deployment            (600 palabras)
  - Seccion 5: API endpoints         (1500 palabras)
  - Seccion 6: Contributing          (300 palabras)

Si embedizas TODO el README como un solo vector:
  -> El vector captura "un README generico sobre un proyecto"
  -> Buscar "autenticacion" retorna TODO el README (5000 palabras)
  -> Claude recibe 5000 palabras cuando solo necesitaba 800

Si divides en chunks por seccion:
  -> Cada seccion tiene su propio vector
  -> Buscar "autenticacion" retorna SOLO la seccion 3 (800 palabras)
  -> Claude recibe exactamente lo que necesita
```

### Estrategias de Chunking

| Estrategia | Ventaja | Desventaja | Usar cuando... |
|------------|---------|------------|----------------|
| **Tamaño fijo** | Simple de implementar | Puede cortar ideas a la mitad | Documentos sin estructura clara |
| **Por párrafos** | Respeta límites naturales | Tamaño variable | Textos narrativos |
| **Por headers** | Secciones coherentes | Solo para markdown/HTML | Documentación técnica |
| **Recursiva** | Mejor balance general | Más compleja | Recomendada por defecto |

In [ ]:
def chunk_by_fixed_size(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Estrategia 1: Tamanio fijo.

    Divide el texto en chunks de N palabras (aprox).
    Simple pero puede cortar ideas a la mitad.

    El overlap incluye las ultimas N palabras del chunk anterior
    al inicio del siguiente, para no perder contexto en los bordes.

    Ejemplo con chunk_size=10, overlap=3:
      Texto: "A B C D E F G H I J K L M N O P"
      Chunk 1: "A B C D E F G H I J"
      Chunk 2: "H I J K L M N O P"  <- "H I J" es el overlap
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap  # retroceder para overlap

    return chunks


def chunk_by_paragraphs(text: str, max_words: int = 500) -> list[str]:
    """
    Estrategia 2: Por parrafos.

    Respeta los limites naturales del texto (parrafos).
    Mejor coherencia pero chunks de tamanio variable.
    Acumula parrafos hasta alcanzar max_words, luego corta.
    """
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        combined = current_chunk + "\n\n" + para if current_chunk else para
        if len(combined.split()) > max_words and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk = combined

    # No olvidar el ultimo chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


def chunk_by_headers(text: str) -> list[str]:
    """
    Estrategia 3: Por headers de markdown.

    Divide en las secciones naturales del documento.
    Ideal para documentacion tecnica que ya tiene estructura.
    Cada seccion (definida por #, ##, ###) se convierte en un chunk.
    """
    import re

    # El patron busca lineas que empiezan con uno o mas '#'
    sections = re.split(r"\n(?=#{1,3} )", text)

    chunks = []
    for section in sections:
        section = section.strip()
        if section and len(section.split()) > 10:  # ignorar secciones muy cortas
            chunks.append(section)

    return chunks


def chunk_recursive(text: str, max_tokens: int = 500) -> list[str]:
    """
    Estrategia 4: Recursiva (la recomendada).

    Intenta dividir por headers. Si un chunk es demasiado grande,
    lo subdivide por parrafos. Si aun es grande, por oraciones.

    Es la estrategia que mejor balancea coherencia y tamanio.
    """
    # Primero dividir por headers
    chunks = chunk_by_headers(text)

    # Si algun chunk es demasiado grande, subdividirlo
    final_chunks = []
    for chunk in chunks:
        words = len(chunk.split())
        if words <= max_tokens:
            final_chunks.append(chunk)
        else:
            # Subdividir por parrafos
            sub_chunks = chunk_by_paragraphs(chunk, max_words=max_tokens)
            final_chunks.extend(sub_chunks)

    return final_chunks


# ---- Demo: comparar estrategias ----

sample_doc = """# Klimbook Authentication

The auth service handles user registration and login.
It uses JWT tokens signed with RS256 for secure authentication.

## Login Flow

The user sends a POST request to /auth/login with email and password.
The service validates credentials using bcrypt hashing.
If valid, it returns an access token (15 min TTL) and a refresh token (7 days TTL).
Refresh tokens are stored in Redis for fast validation and revocation.

## Token Validation

Every request to protected endpoints must include the JWT in the Authorization header.
The API gateway validates the token signature using the public key.
If expired, the client can use the refresh token to get a new access token.

## Security Considerations

Passwords are hashed with bcrypt using a cost factor of 12.
Rate limiting is applied to the login endpoint (5 attempts per minute).
Failed login attempts are logged for security monitoring.
"""

print("=== Comparacion de Estrategias ===\n")
for name, func in [
    ("Fixed (100 words)", lambda t: chunk_by_fixed_size(t, 100, 20)),
    ("Paragraphs (100 words)", lambda t: chunk_by_paragraphs(t, 100)),
    ("Headers", chunk_by_headers),
    ("Recursive (100)", lambda t: chunk_recursive(t, 100)),
]:
    chunks = func(sample_doc)
    print(f"{name}: {len(chunks)} chunks")
    for i, c in enumerate(chunks):
        print(f"  [{i}] ({len(c.split())} words) {c[:60]}...")
    print()

### Metadata Enrichment

Cuando guardas un chunk, también guardas **metadata** sobre de dónde vino. Esto es importante para:

1. **Darle contexto a Claude** sobre la fuente (ej: *"según docs/ARCHITECTURE.md..."*)
2. **Filtrar resultados** por tipo de documento (ej: *"solo buscar en API docs"*)

In [ ]:
# Cada chunk se guarda con su metadata
chunk_with_metadata = {
    "text": "The auth service uses JWT tokens with RS256...",
    "metadata": {
        "source_file": "docs/ARCHITECTURE.md",
        "section": "Authentication",
        "doc_type": "architecture",
        "chunk_index": 3,
        "total_chunks": 12,
    },
}

# Cuando recuperas chunks, puedes filtrar por metadata:
#   "Solo buscar en documentos de tipo 'api_docs'"
#   "Solo buscar en archivos modificados despues de 2026-01-01"

print("Chunk con metadata:")
for key, value in chunk_with_metadata.items():
    print(f"  {key}: {value}")

---

## 11.5 Vector Stores: Dónde se Guardan los Embeddings

Un **vector store** es una base de datos especializada en almacenar vectores y buscar los más similares a un vector dado. Es como una base de datos normal pero en vez de buscar por igualdad (`WHERE name = 'X'`), buscas por **similitud** (los K vectores más cercanos a este).

| Opción | Tipo | Ideal para | Ventaja |
|--------|------|------------|---------|
| **ChromaDB** | In-process / local | Desarrollo y prototyping | Sin infraestructura externa |
| **pgvector** | Extensión PostgreSQL | Producción | Se integra con tu DB existente |
| FAISS | Librería (Meta) | Datasets masivos | Muy rápido para millones de vectores |
| Pinecone | SaaS | Producción sin ops | Fully managed, escala automática |

### Opción 1: ChromaDB (recomendado para desarrollo)

ChromaDB es un vector store ligero que corre en Python. Ideal para prototyping porque no necesitas instalar nada externo.

In [ ]:
# pip install chromadb

import chromadb
from sentence_transformers import SentenceTransformer

# ---- SETUP ----

# Cliente persistente (guarda en disco).
# Si usas Client() sin persist_directory, los datos se pierden al cerrar.
chroma_client = chromadb.PersistentClient(path="./klimbook_vectorstore")

# Crear una coleccion (equivalente a una "tabla" en SQL).
# get_or_create: si ya existe, la abre; si no, la crea.
collection = chroma_client.get_or_create_collection(
    name="klimbook_docs",
    metadata={"description": "Documentacion de Klimbook"},
)

# Modelo de embeddings
embed_model = SentenceTransformer("all-MiniLM-L6-v2")


# ---- INDEXAR DOCUMENTOS ----

def index_document(filepath: str, doc_type: str = "docs"):
    """
    Indexa un documento en el vector store.

    1. Lee el archivo
    2. Lo divide en chunks (usando chunk_recursive)
    3. Genera embeddings para cada chunk
    4. Los guarda en ChromaDB con metadata
    """
    with open(filepath) as f:
        text = f.read()

    # Dividir en chunks
    chunks = chunk_recursive(text, max_tokens=300)

    if not chunks:
        print(f"  No chunks para {filepath}")
        return

    # Generar embeddings para todos los chunks de una vez
    embeddings = embed_model.encode(chunks).tolist()

    # IDs unicos para cada chunk.
    # ChromaDB requiere un ID unico por documento.
    filename = filepath.split("/")[-1]
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]

    # Metadata para cada chunk
    metadatas = [
        {
            "source_file": filepath,
            "doc_type": doc_type,
            "chunk_index": i,
            "total_chunks": len(chunks),
        }
        for i in range(len(chunks))
    ]

    # Agregar al vector store.
    # Pasamos embeddings pre-calculados (mas eficiente que dejar
    # que ChromaDB los calcule internamente).
    collection.add(
        ids=ids,
        documents=chunks,       # texto original (para recuperarlo)
        embeddings=embeddings,  # los vectores
        metadatas=metadatas,    # metadata adicional
    )

    print(f"  Indexado: {filepath} -> {len(chunks)} chunks")


# Ejemplo: indexar la documentacion de Klimbook
# (estos archivos no existen, es solo para mostrar el patron)
docs = [
    ("docs/README.md", "readme"),
    ("docs/ARCHITECTURE.md", "architecture"),
    ("docs/API_DOCS.md", "api"),
    ("docs/DEPLOYMENT.md", "deployment"),
    ("docs/CHANGELOG.md", "changelog"),
    ("src/auth_service/README.md", "service"),
    ("src/climbing_service/README.md", "service"),
]

# for filepath, doc_type in docs:
#     index_document(filepath, doc_type)
# print(f"\nTotal chunks indexados: {collection.count()}")

print("Funcion index_document definida.")
print(f"Coleccion '{collection.name}' lista ({collection.count()} chunks actuales).")

### Búsqueda en ChromaDB

In [ ]:
def search_docs(query: str, top_k: int = 5, doc_type: str = None) -> list[dict]:
    """
    Busca los chunks mas relevantes para una query.

    1. Convierte la query a un embedding
    2. Busca los top_k chunks mas similares en ChromaDB
    3. Retorna los chunks con texto, metadata, y score de similitud

    Args:
        query: Pregunta del usuario en lenguaje natural
        top_k: Cuantos resultados retornar (default: 5)
        doc_type: Filtrar por tipo de documento (opcional)
    """
    # Embedizar la query
    query_embedding = embed_model.encode([query]).tolist()

    # Filtro de metadata (opcional)
    where_filter = None
    if doc_type:
        where_filter = {"doc_type": doc_type}

    # ChromaDB calcula la distancia coseno internamente
    # y retorna los top_k resultados mas cercanos.
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )

    # Formatear resultados.
    # ChromaDB retorna listas anidadas porque soporta multiple queries.
    # Como solo hacemos una query, tomamos [0].
    chunks = []
    for i in range(len(results["ids"][0])):
        chunks.append(
            {
                "id": results["ids"][0][i],
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                # ChromaDB retorna distancia (menor = mas similar).
                # Convertimos a similitud (mayor = mas similar).
                "similarity": 1 - results["distances"][0][i],
            }
        )

    return chunks


print("Funcion search_docs definida.")
print("Uso: search_docs('how does auth work?', top_k=5)")
print("     search_docs('deployment', doc_type='deployment')")

### Opción 2: pgvector (recomendado para producción)

**pgvector** es una extensión de PostgreSQL. Como ya usas PostgreSQL para Klimbook, puedes guardar los embeddings en la **misma base de datos** sin agregar infraestructura.

La ventaja principal: puedes combinar **búsqueda vectorial con filtros SQL normales**.

In [ ]:
# ---- pgvector: Setup SQL ----
# (ejecutar en PostgreSQL directamente)

pgvector_setup_sql = """
-- Instalar la extension
CREATE EXTENSION IF NOT EXISTS vector;

-- Crear tabla para los chunks
CREATE TABLE doc_chunks (
    id SERIAL PRIMARY KEY,
    text TEXT NOT NULL,
    embedding VECTOR(384),  -- 384 dimensiones para all-MiniLM-L6-v2
    source_file VARCHAR(200),
    doc_type VARCHAR(50),
    chunk_index INTEGER,
    created_at TIMESTAMP DEFAULT NOW()
);

-- Indice IVFFlat para busqueda rapida.
-- Divide el espacio vectorial en "clusters". Cuando buscas,
-- solo compara con los clusters mas cercanos (no TODOS los vectores).
-- lists=100: 100 clusters. Para <10,000 vectores, 100 esta bien.
CREATE INDEX idx_chunks_embedding
ON doc_chunks USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
"""

print(pgvector_setup_sql)


# ---- pgvector: Busqueda en Python ----

pgvector_search_code = '''
import asyncpg

async def search_pgvector(query_embedding: list[float], top_k: int = 5):
    """
    Busca los chunks mas similares usando pgvector.
    El operador <=> calcula distancia coseno.
    ORDER BY distancia ASC retorna los mas similares primero.
    """
    async with pool.acquire() as conn:
        rows = await conn.fetch(
            """
            SELECT
                text,
                source_file,
                doc_type,
                1 - (embedding <=> $1::vector) AS similarity
            FROM doc_chunks
            ORDER BY embedding <=> $1::vector
            LIMIT $2
            """,
            str(query_embedding),  # pgvector espera string '[0.1, 0.2, ...]'
            top_k,
        )
    return [dict(row) for row in rows]


# Ventaja: combinar vectores con filtros SQL normales
filtered_query = """
SELECT text, 1 - (embedding <=> $1::vector) AS similarity
FROM doc_chunks
WHERE doc_type = 'api'
AND created_at > '2026-01-01'
ORDER BY embedding <=> $1::vector
LIMIT 5;
"""
'''

print("\n-- Busqueda con pgvector en Python:")
print(pgvector_search_code)

---

## 11.6 El Pipeline RAG Completo

Ahora juntamos todo: **embeddings + chunking + vector store + Claude**.

El agente tiene una tool `search_docs` que busca en la documentación. Cuando el usuario pregunta algo sobre Klimbook, Claude llama a la tool, recibe los chunks relevantes, y genera una respuesta basada en ellos.

```
Usuario: "Como funciona la autenticacion?"
    |
    v
Claude decide: necesito buscar informacion
    |
    v
Tool call: search_docs(query="authentication JWT login")
    |
    v
ChromaDB retorna top-5 chunks relevantes
    |
    v
Claude recibe los chunks como tool_result
    |
    v
Claude genera respuesta citando la documentacion
```

In [ ]:
import json
from anthropic import Anthropic

# ---- Setup ----

client = Anthropic()
MODEL = "claude-sonnet-4-20250514"


# ---- Tool: search_docs (la misma de arriba, adaptada para el agente) ----

def search_docs_tool(query: str, top_k: int = 5) -> dict:
    """
    Tool del agente: busca documentacion relevante.

    Esta funcion se registra como tool en el agente.
    Cuando Claude necesita informacion sobre Klimbook,
    llama a esta tool con la query relevante.
    """
    # 1. Embedizar la query
    query_embedding = embed_model.encode([query]).tolist()

    # 2. Buscar en ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # 3. Formatear resultados (filtrar por similitud minima)
    chunks = []
    for i in range(len(results["ids"][0])):
        similarity = 1 - results["distances"][0][i]

        # Solo incluir chunks con similitud razonable.
        # Similitud < 0.3 = chunk no es relevante, solo agrega ruido.
        if similarity < 0.3:
            continue

        chunks.append(
            {
                "text": results["documents"][0][i],
                "source": results["metadatas"][0][i].get("source_file", "unknown"),
                "similarity": round(similarity, 3),
            }
        )

    return {
        "query": query,
        "results_count": len(chunks),
        "chunks": chunks,
    }


# ---- Definicion de la tool para la API de Anthropic ----

TOOLS = [
    {
        "name": "search_docs",
        "description": (
            "Searches Klimbook's documentation and codebase for relevant "
            "information. Use this whenever you need specific details about "
            "how Klimbook works, its architecture, API endpoints, deployment, "
            "or any technical aspect. Returns the most relevant text chunks "
            "with their source files and similarity scores."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query describing what you're looking for.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of results to return (default: 5, max: 10).",
                    "default": 5,
                },
            },
            "required": ["query"],
        },
    },
]

TOOL_FUNCTIONS = {"search_docs": search_docs_tool}

SYSTEM = """You are a technical support assistant for Klimbook, a social
network for rock climbers.

You have access to Klimbook's documentation through the search_docs tool.
ALWAYS use search_docs before answering technical questions about Klimbook.
Do NOT make up information -- only use what the documentation says.

If the documentation doesn't contain the answer, say so honestly.
Cite the source file when referencing specific documentation."""


# ---- Agente RAG ----

def ask_rag(question: str) -> str:
    """
    Hace una pregunta al agente RAG.

    Flujo tipico:
    1. Usuario pregunta sobre Klimbook
    2. Claude llama search_docs() para buscar en la documentacion
    3. Claude recibe los chunks relevantes como tool_result
    4. Claude genera una respuesta basada en los chunks
    """
    messages = [{"role": "user", "content": question}]

    for iteration in range(5):  # max 5 tool calls
        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            tools=TOOLS,
            system=SYSTEM,
            messages=messages,
        )

        # Respuesta final
        if response.stop_reason == "end_turn":
            return response.content[0].text

        # Tool use: acumular resultado y seguir el loop
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    func = TOOL_FUNCTIONS[block.name]
                    result = func(**block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result, ensure_ascii=False),
                        }
                    )

            messages.append({"role": "user", "content": tool_results})

    return "No pude encontrar la respuesta."


# ---- Ejemplo de uso (requiere docs indexados) ----
# answer = ask_rag("Como funciona la autenticacion en Klimbook?")
# print(answer)

print("Agente RAG definido: ask_rag(question)")
print("Uso: ask_rag('Como funciona la autenticacion en Klimbook?')")

---

## 11.7 Evaluación de la Calidad del RAG

¿Cómo saber si tu RAG funciona bien? Necesitas medir **dos cosas**:

| Métrica | Qué mide | Cómo |
|---------|----------|------|
| **Retrieval quality** | ¿Los chunks recuperados son relevantes? | Recall y Precision |
| **Answer quality** | ¿La respuesta final es correcta y específica? | Comparación con/sin RAG |

### 1. Relevancia de los chunks recuperados

| Métrica | Fórmula | Interpretación |
|---------|---------|----------------|
| **Recall** | (archivos encontrados ∩ esperados) / esperados | ¿Encontramos todo lo relevante? |
| **Precision** | (archivos encontrados ∩ esperados) / encontrados | ¿Todo lo que encontramos es relevante? |

> **Ideal**: Recall alto (no perderse nada) + Precision alta (no traer basura). En la práctica, subir uno baja el otro -- hay que encontrar el balance.

In [ ]:
def evaluate_retrieval(test_cases: list[dict]):
    """
    Evalua la calidad de la recuperacion.

    Cada test case tiene:
    - query: la pregunta
    - expected_sources: archivos que deberian aparecer en los resultados

    Mide:
    - Recall: que % de los archivos esperados aparecen en los resultados
    - Precision: que % de los resultados son archivos esperados
    """
    total_recall = 0
    total_precision = 0

    for case in test_cases:
        results = search_docs(case["query"], top_k=5)

        retrieved_sources = {r["metadata"]["source_file"] for r in results}
        expected_sources = set(case["expected_sources"])

        # Recall: de los archivos esperados, cuantos encontramos?
        if expected_sources:
            found = retrieved_sources & expected_sources
            recall = len(found) / len(expected_sources)
        else:
            recall = 1.0

        # Precision: de los resultados, cuantos son relevantes?
        if retrieved_sources:
            relevant = retrieved_sources & expected_sources
            precision = len(relevant) / len(retrieved_sources)
        else:
            precision = 0.0

        total_recall += recall
        total_precision += precision

        print(f"  Query: {case['query'][:50]}")
        print(f"    Recall: {recall:.2f} | Precision: {precision:.2f}")
        print(f"    Found: {retrieved_sources}")

    n = len(test_cases)
    print(f"\n  Average Recall:    {total_recall / n:.2f}")
    print(f"  Average Precision: {total_precision / n:.2f}")


# Test cases para Klimbook
test_cases = [
    {
        "query": "How does authentication work?",
        "expected_sources": ["docs/ARCHITECTURE.md", "src/auth_service/README.md"],
    },
    {
        "query": "How to deploy to production?",
        "expected_sources": ["docs/DEPLOYMENT.md"],
    },
    {
        "query": "What endpoints does the climbing service have?",
        "expected_sources": ["docs/API_DOCS.md", "src/climbing_service/README.md"],
    },
    {
        "query": "What changed in version 2.9.0?",
        "expected_sources": ["docs/CHANGELOG.md"],
    },
]

# Descomentar cuando tengas docs indexados:
# evaluate_retrieval(test_cases)

print("Funcion evaluate_retrieval definida.")
print(f"Test cases preparados: {len(test_cases)}")

### 2. Calidad de las respuestas: con RAG vs sin RAG

La forma más directa de evaluar es **comparar lado a lado** las respuestas de Claude con y sin acceso a la documentación:

In [ ]:
def compare_with_without_rag(questions: list[str]):
    """
    Compara respuestas de Claude con y sin RAG.
    Imprime ambas lado a lado para evaluacion manual.
    """
    for q in questions:
        # Sin RAG: Claude responde solo con su conocimiento general
        response_no_rag = client.messages.create(
            model=MODEL,
            max_tokens=500,
            messages=[{"role": "user", "content": q}],
        )

        # Con RAG: Claude usa los documentos via search_docs
        response_rag = ask_rag(q)

        print(f"\n{'=' * 60}")
        print(f"  Q: {q}")
        print(f"{'=' * 60}")
        print(f"\n  SIN RAG:")
        print(f"  {response_no_rag.content[0].text[:300]}")
        print(f"\n  CON RAG:")
        print(f"  {response_rag[:300]}")


# Ejemplo (requiere docs indexados y API key):
# compare_with_without_rag([
#     "Como funciona la autenticacion en Klimbook?",
#     "Que endpoints tiene el servicio de escalada?",
#     "Como se despliega Klimbook en produccion?",
# ])

print("Funcion compare_with_without_rag definida.")

---

## 11.8 Tips Avanzados

### 1. Reranking: mejorar los resultados de búsqueda

La búsqueda vectorial a veces trae resultados que son **semánticamente cercanos pero no realmente relevantes**. Un reranker los reordena con un modelo más preciso.

```
Flujo con reranking:
  1. Buscar top-20 con embeddings     (rapido pero impreciso)
  2. Reranker reduce a top-5          (lento pero preciso)
```

> **Nota**: El reranker puede ser un modelo especializado (como `cross-encoder/ms-marco-MiniLM-L6`) o incluso Claude Haiku como evaluador barato y rápido.

In [ ]:
def rerank_with_llm(query: str, chunks: list[dict], top_k: int = 5) -> list[dict]:
    """
    Usa Haiku para reranquear chunks por relevancia.

    Flujo:
    1. Recibe top-20 chunks de la busqueda vectorial
    2. Le pide a Haiku que ordene por relevancia real
    3. Retorna solo los top-k mas relevantes

    Haiku es barato (~$0.25/M tokens) y rapido, ideal para este paso.
    """
    chunks_text = "\n\n".join(
        f"[{i}] {c['text'][:200]}" for i, c in enumerate(chunks)
    )

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        system="You rank text chunks by relevance to a query.",
        messages=[
            {
                "role": "user",
                "content": (
                    f"Query: {query}\n\nChunks:\n{chunks_text}\n\n"
                    f"Return the indices of the {top_k} most relevant "
                    f"chunks as a JSON array, most relevant first: [3, 0, 7, ...]"
                ),
            }
        ],
        temperature=0.0,
    )

    indices = json.loads(response.content[0].text)
    return [chunks[i] for i in indices if i < len(chunks)]


print("Funcion rerank_with_llm definida.")
print("Uso: reranked = rerank_with_llm('auth flow', raw_chunks, top_k=5)")

### 2. Hybrid Retrieval: búsqueda semántica + keywords

A veces la búsqueda semántica falla con **términos muy específicos** (nombres de funciones, IDs, versiones). Combinar con búsqueda por keywords mejora los resultados.

| Tipo | Fortaleza | Debilidad | Ejemplo |
|------|-----------|-----------|---------|
| **Semántica** | Entiende sinónimos y contexto | Falla con nombres exactos | "autenticación" encuentra "login flow" |
| **Keywords** | Coincidencia exacta | No entiende sinónimos | "PreferencesResponse" encuentra la clase exacta |
| **Híbrida** | Lo mejor de ambas | Más compleja | Cubre ambos casos |

In [ ]:
def hybrid_search(query: str, top_k: int = 5) -> list[dict]:
    """
    Combina busqueda semantica (embeddings) con busqueda por keywords.

    Resuelve el caso donde alguien busca "PreferencesResponse":
    - Busqueda semantica podria no entender que es un nombre de clase
    - Busqueda por keywords lo encuentra exacto
    """
    # 1. Busqueda semantica (embeddings custom)
    semantic_results = search_docs(query, top_k=top_k * 2)

    # 2. Busqueda por keywords (ChromaDB busca en el texto)
    keyword_results = collection.query(
        query_texts=[query],  # ChromaDB hace embedding internamente
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    # 3. Combinar y deduplicar
    seen_ids = set()
    combined = []

    for result in semantic_results:
        rid = result.get("id", result["text"][:50])
        if rid not in seen_ids:
            combined.append(result)
            seen_ids.add(rid)

    # Agregar resultados de keywords que no esten ya.
    # En produccion usarias Reciprocal Rank Fusion (RRF) para
    # combinar los scores de forma mas sofisticada.

    return combined[:top_k]


print("Funcion hybrid_search definida.")
print("Nota: en produccion, usar Reciprocal Rank Fusion (RRF)")
print("para combinar scores de busqueda semantica y keywords.")

### 3. ¿Cuántos chunks recuperar (top_k)?

| top_k | Contexto | Costo | Riesgo |
|-------|----------|-------|--------|
| **3** | Poco | Bajo | Perder información relevante |
| **5** | Balanceado | Medio | **Recomendado como default** |
| **8-10** | Mucho | Alto | Incluir chunks irrelevantes que confundan a Claude |

> **Regla general**: Empieza con `top_k=5`. Si las respuestas son incompletas, sube a 8-10. Si son buenas pero costosas, baja a 3.

### 4. Problemas comunes y soluciones

| Problema | Síntoma | Solución |
|----------|---------|----------|
| Chunks muy grandes | Respuestas vagas, mucho contexto irrelevante | Reducir `max_tokens` en chunking a 200-300 |
| Chunks muy pequeños | Pierde contexto, respuestas fragmentadas | Aumentar `max_tokens` o usar overlap |
| Embeddings de baja calidad | Retrieval trae chunks irrelevantes | Cambiar a un modelo de embeddings mejor (Voyage AI) |
| Documentación desactualizada | Respuestas con info vieja | Implementar re-indexación periódica |
| Queries ambiguas | Claude busca algo diferente a lo que el usuario quiere | Hacer que Claude reformule la query antes de buscar |

---

## Resumen Semana 11

### Arquitectura RAG

```
 Tus Documentos          Vector Store           Agente + Claude
 ==============         =============          ================
                                              
  README.md    --->  chunk + embed  --->  Guardar vectores
  API_DOCS.md  --->  chunk + embed  --->  en ChromaDB/pgvector
  CHANGELOG.md --->  chunk + embed  --->       |
  codigo.py    --->  chunk + embed  --->       |
                                               |
  (FASE 1: Indexacion -- se hace una vez)      |
                                               |
  =============================================|============
                                               |
  Pregunta del usuario                         |
       |                                       |
       v                                       |
  Claude llama search_docs(query)              |
       |                                       |
       v                                       |
  Embed query --> buscar top-K similares  <----+
       |
       v
  Inyectar chunks en el contexto
       |
       v
  Claude genera respuesta especifica
  
  (FASE 2: Consulta -- cada pregunta)
```

### Conceptos Clave

| Concepto | Qué es | Herramienta usada |
|----------|--------|-------------------|
| **Embedding** | Vector numérico que captura significado semántico del texto | `sentence-transformers`, Voyage AI |
| **Cosine Similarity** | Métrica de similitud entre vectores (0.0 a 1.0) | `numpy` |
| **Chunking** | Dividir documentos grandes en fragmentos manejables | Funciones propias (recursive best) |
| **Vector Store** | BD especializada en buscar vectores similares | ChromaDB (dev), pgvector (prod) |
| **Retrieval** | Buscar los top-K chunks más relevantes para una query | `collection.query()` |
| **Reranking** | Reordenar resultados con un modelo más preciso | Claude Haiku, cross-encoder |
| **Hybrid Search** | Combinar búsqueda semántica + keywords | Embeddings + text match |

### Checklist de Implementación

```
[ ] Elegir modelo de embeddings (sentence-transformers para dev, Voyage para prod)
[ ] Implementar chunking recursivo con metadata
[ ] Configurar vector store (ChromaDB local o pgvector en PostgreSQL)
[ ] Indexar todos los documentos del proyecto
[ ] Crear tool search_docs para el agente
[ ] Configurar system prompt para que Claude SIEMPRE busque antes de responder
[ ] Escribir test cases de evaluacion (query + expected_sources)
[ ] Medir recall y precision del retrieval
[ ] Comparar respuestas con/sin RAG para verificar mejora
[ ] Considerar reranking si la precision es baja
[ ] Considerar hybrid search si fallan queries con nombres exactos
```

### Modelo Mental

```
"Cuando necesites que Claude sepa sobre TU proyecto:
 1. No metas todo en el system prompt (limite de tokens, no escala)
 2. No hagas fine-tuning (caro, rigido, pierde generalidad)
 3. Usa RAG: busca lo relevante, inyecta solo eso, Claude responde"
```